In [0]:
%sql
SELECT * FROM marathos.gold.dim_events
WHERE host_country_code = 'SWE' AND event_start_date != event_end_date
LIMIT 20;

In [0]:
%sql
SELECT 
    event_name,
    event_unit_type,
    event_distance_value,
    event_distance_km,
    host_country_code
FROM marathos.silver.marathon_results
WHERE event_unit_type = 'mi'
LIMIT 5

In [0]:
# 

from pyspark.sql.functions import col, count, when, isnan
 
# Verify dim_events
df = spark.sql("SELECT * FROM marathos.gold.dim_events")

# 1. count unique events
print("number of unique events:", df.count())

# 2. check duplicates
print("number of duplicates on event_id:", df.count() - df.dropDuplicates(["event_id"]).count())

# 3 Verify miles conversion
print("\nMiles events sample:")
df.filter(col("event_unit_type") == "mi").select(
    "event_name", "event_distance_value", "event_distance_km"
).show(5)

# 4. Verify multi-day events
print("\nmulti-day event:")
df.filter(col("event_start_date") != col("event_end_date")).select(
    "event_name", "event_start_date", "event_end_date"
).show(5)

# 5. Validate null values in key columns
print("\nNull-values per column:")
df.select([
    count(when(col(c).isNull(), c)).alias(c) 
    for c in ["event_id", "event_name", "event_unit_type", "host_country_code"]
]).show()

In [0]:
%sql
SELECT * 
FROM marathos.gold.mart_swedish_races
WHERE event_unit_type = 'km'
LIMIT 20;

In [0]:
%sql
-- verify marts_events_calendar
SELECT *
FROM marathos.gold.mart_events_calendar
LIMIT 5